In [1]:
# ── Cài đặt dependencies ─────────────────────────────────────────
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install numpy seaborn pandas matplotlib tqdm scikit-learn -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
"""
Stage 1 — ResNet-1D Feature Extractor
======================================
Fold split: SUBJECT-WISE, khớp 100% với TCN notebook (Cell 4).

Logic chia fold của TCN (Cell 4):
    rng          = np.random.default_rng(seed=123)
    shuffled_ids = rng.permutation(subject_ids).tolist()
    fold_groups  = np.array_split(shuffled_ids, N_FOLDS)

Script này tái tạo chính xác chuỗi trên để fold i của ResNet
và fold i của TCN luôn dùng CÙNG tập train/test subjects.

Checkpoint được lưu: saved_models/resnet_fold{i+1}_best.pth
  → TCN Cell 12 load bằng: CKPT_DIR / f'resnet_fold{fold_i+1}.pth'
  → Đổi tên hoặc sửa CKPT_DIR trong TCN cho khớp tuỳ ý.
"""

import os
import copy
import glob
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score as sk_f1

# ══════════════════════════════════════════════════════════════════
# CONFIG — phải khớp với TCN notebook (Cell 2)
# ══════════════════════════════════════════════════════════════════
N_FOLDS       = 10
N_CLASSES     = 5
EPOCH_SAMPLES = 3000          # 30 s × 100 Hz
N_FEAT        = 128

RESNET_LR         = 1e-3
RESNET_BATCH_SIZE = 64
RESNET_EPOCHS     = 40
RESNET_PATIENCE   = 8         # early stopping theo val Macro-F1
RESNET_VAL_RATIO  = 0.15      # tỉ lệ val split ở mức RECORD

SEED = 123                    # dùng CÙNG seed với TCN

DATA_DIR  = Path('./data')
OUT_DIR   = Path('/workspace/data')
CKPT_DIR  = OUT_DIR / 'saved_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


# ══════════════════════════════════════════════════════════════════
# 0. REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"Global seed set to {seed}")


# ══════════════════════════════════════════════════════════════════
# 1. KIẾN TRÚC RESNET-1D (giữ nguyên so với TCN Cell 5)
# ══════════════════════════════════════════════════════════════════
class ResNetBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, 7, stride=stride, padding=3, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, 7, stride=1, padding=3, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x):
        res = self.shortcut(x)
        x   = self.relu(self.bn1(self.conv1(x)))
        x   = self.bn2(self.conv2(x))
        return self.relu(x + res)


class EEG_ResNet1D(nn.Module):
    def __init__(self, feature_dim=128, num_classes=5):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 50, stride=5, padding=25, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(3, stride=2, padding=1),
        )
        self.layer1 = ResNetBlock1D(32,  64,         stride=1)
        self.layer2 = ResNetBlock1D(64,  128,        stride=2)
        self.layer3 = ResNetBlock1D(128, feature_dim,stride=2)
        self.gap     = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=0.5)
        self.fc      = nn.Linear(feature_dim, num_classes)

    def forward(self, x, extract_features=False):
        x        = self.stem(x)
        x        = self.layer1(x)
        x        = self.layer2(x)
        x        = self.layer3(x)
        features = self.gap(x).squeeze(-1)      # (B, feature_dim)
        if extract_features:
            return features
        return self.fc(self.dropout(features))


# ══════════════════════════════════════════════════════════════════
# 2. DATASET
# ══════════════════════════════════════════════════════════════════
class _SimpleEEGDataset(Dataset):
    def __init__(self, signals: np.ndarray, labels: np.ndarray):
        self.signals = signals          # (N, 3000)  float32
        self.labels  = labels           # (N,)        int64

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.signals[idx].astype(np.float32)).unsqueeze(0)  # (1,3000)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


# ══════════════════════════════════════════════════════════════════
# 3. LOAD DỮ LIỆU — khớp với TCN Cell 3
# ══════════════════════════════════════════════════════════════════
def normalize_labels(y: np.ndarray) -> np.ndarray:
    """Đảm bảo label range 0-4; nhãn ngoài phạm vi → -1 (rác)."""
    y = y.astype(np.int64).ravel()
    if y.min() == 1 and y.max() <= 5:
        y = y - 1          # shift 1-5 → 0-4 (AASM convention)
    y[(y < 0) | (y >= N_CLASSES)] = -1
    return y


def load_npz(fpath: Path):
    """
    Tái tạo chính xác hàm load_npz của TCN (Cell 3):
      - Normalize labels
      - Trim: giữ 60 epoch trước/sau vùng ngủ
    """
    d    = np.load(str(fpath), allow_pickle=True)
    sigs = d['x'].astype(np.float32)
    lbls = normalize_labels(d['y'])

    if sigs.ndim == 3:
        sigs = sigs[:, 0, :]              # lấy kênh đầu tiên
    if sigs.ndim == 2 and sigs.shape[1] != EPOCH_SAMPLES:
        sigs = sigs.T                     # (3000, T) → (T, 3000)

    sleep_idx = np.where((lbls >= 1) & (lbls <= 4))[0]
    if len(sleep_idx) > 0:
        s = max(0, sleep_idx[0]  - 60)
        e = min(len(lbls), sleep_idx[-1] + 61)
        sigs, lbls = sigs[s:e], lbls[s:e]

    assert sigs.shape[1] == EPOCH_SAMPLES, f'shape lỗi: {sigs.shape}'
    return sigs, lbls


def load_all_records(data_dir: Path):
    """
    Trả về subject_records và all_records — cùng cấu trúc với TCN Cell 3.
    subject ID: ký tự 3-4 của tên file (SC4001E0 → '00').
    """
    npz_files = sorted(data_dir.glob('SC*.npz'))
    if not npz_files:
        raise FileNotFoundError(f'Không tìm thấy SC*.npz trong {data_dir}')
    print(f'Tìm thấy {len(npz_files)} NPZ files')

    subject_records: dict[str, list] = defaultdict(list)
    all_records: list = []

    for fpath in npz_files:
        try:
            sigs, lbls = load_npz(fpath)
            sid  = fpath.stem[3:5]          # khớp với TCN Cell 3
            rec  = {'sid': sid, 'fname': fpath.stem, 'sigs': sigs, 'lbls': lbls}
            subject_records[sid].append(rec)
            all_records.append(rec)
        except Exception as e:
            print(f'  ✗ {fpath.name}: {e}')

    print(f'✓ {len(all_records)} records  |  {len(subject_records)} subjects\n')
    return subject_records, all_records


# ══════════════════════════════════════════════════════════════════
# 4. FOLD SPLIT — tái tạo 100% Cell 4 của TCN
# ══════════════════════════════════════════════════════════════════
def build_fold_groups(subject_records: dict, n_folds: int = N_FOLDS, seed: int = SEED):
    """
    Tái tạo Cell 4 của TCN:
        rng          = np.random.default_rng(seed=123)
        shuffled_ids = rng.permutation(subject_ids).tolist()
        fold_groups  = np.array_split(shuffled_ids, N_FOLDS)
    """
    subject_ids  = sorted(subject_records.keys())
    rng          = np.random.default_rng(seed=seed)         # ← CÙNG seed & RNG class
    shuffled_ids = rng.permutation(subject_ids).tolist()    # ← CÙNG permutation
    fold_groups  = np.array_split(shuffled_ids, n_folds)    # ← CÙNG split

    print(f'Subject-wise {n_folds}-fold CV (seed={seed}):')
    for i, grp in enumerate(fold_groups):
        sids  = list(grp)
        n_rec = sum(len(subject_records[s]) for s in sids)
        n_ep  = sum(len(r['lbls']) for s in sids for r in subject_records[s])
        print(f'  Fold {i+1:2d}: {sids}  {n_rec} rec  {n_ep:,} epochs')
    print()
    return fold_groups


# ══════════════════════════════════════════════════════════════════
# 5. TRAIN ResNet-1D — khớp với TCN Cell 8 (train_resnet_for_fold)
# ══════════════════════════════════════════════════════════════════
def _collect_valid(records: list):
    """Gom signals + labels hợp lệ (lọc nhãn -1) từ danh sách records."""
    sigs_all, lbls_all = [], []
    for rec in records:
        valid = rec['lbls'] != -1
        sigs_all.append(rec['sigs'][valid])
        lbls_all.append(rec['lbls'][valid])
    return np.concatenate(sigs_all), np.concatenate(lbls_all)


def train_resnet_for_fold(
    train_records: list,
    fold_idx: int,
    device: torch.device,
    verbose: bool = True,
) -> nn.Module:
    """
    Huấn luyện ResNet-1D với val-split (15%) + early stopping theo val Macro-F1.
    Khớp logic với TCN Cell 8 (train_resnet_for_fold).

    val_split được lấy từ epoch level (stratified) — tách TRƯỚC KHI
    dựng DataLoader, scaler không cần ở bước này vì TCN nhận
    raw 128-dim features rồi project qua input_proj của chính nó.
    """
    # ── [FIX] Tách val ở mức RECORD, không phải epoch ───────────────
    # Epoch-level split (train_test_split trên X_all) gây data leakage:
    # các epoch từ cùng một bản ghi (đêm ngủ) có thể xuất hiện ở cả
    # train lẫn val.  Vì EEG epochs trong một đêm tương quan thời gian
    # rất cao, val MF1 sẽ bị inflate → early stopping không đáng tin.
    # Giải pháp: chia records trước, rồi mới pool epochs riêng biệt.
    n_val_recs     = max(1, round(len(train_records) * RESNET_VAL_RATIO))
    rng_rec        = np.random.default_rng(42 + fold_idx)
    idx_perm       = rng_rec.permutation(len(train_records)).tolist()
    val_recs_local = [train_records[i] for i in idx_perm[:n_val_recs]]
    tr_recs_local  = [train_records[i] for i in idx_perm[n_val_recs:]]

    X_train, y_train = _collect_valid(tr_recs_local)
    X_val,   y_val   = _collect_valid(val_recs_local)

    # ── Class weights (fit trên train split) ──────────────────────────
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    cw      = torch.FloatTensor(weights).to(device)
    if verbose:
        print(f'    class_weights: {np.round(weights, 3)}')
        print(f'    Train {len(y_train):,} epochs ({len(tr_recs_local)} recs)  |  Val {len(y_val):,} epochs ({len(val_recs_local)} recs)')

    g_train = torch.Generator()
    g_train.manual_seed(SEED + fold_idx)

    tr_ld = DataLoader(
        _SimpleEEGDataset(X_train, y_train),
        batch_size=RESNET_BATCH_SIZE, shuffle=True,
        generator=g_train,
        worker_init_fn=lambda wid: np.random.seed(SEED + fold_idx + wid),
        num_workers=2, pin_memory=(device.type == 'cuda'),
    )
    vl_ld = DataLoader(
        _SimpleEEGDataset(X_val, y_val),
        batch_size=RESNET_BATCH_SIZE * 2, shuffle=False,
        num_workers=2, pin_memory=(device.type == 'cuda'),
    )

    # ── Model, loss, optimizer ────────────────────────────────────────
    model     = EEG_ResNet1D(feature_dim=N_FEAT, num_classes=N_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.AdamW(model.parameters(), lr=RESNET_LR, weight_decay=1e-4)

    best_mf1   = -1.0
    no_improve = 0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(RESNET_EPOCHS):
        # Train
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        for bx, by in tr_ld:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            out  = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item()
            tr_correct += (out.argmax(1) == by).sum().item()
            tr_total   += by.size(0)

        # Validate
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for vx, vy in vl_ld:
                out = model(vx.to(device))
                preds.extend(out.argmax(1).cpu().numpy())
                trues.extend(vy.numpy())
        val_mf1 = sk_f1(trues, preds, average='macro', zero_division=0)

        if verbose and (epoch + 1) % 5 == 0:
            is_best = val_mf1 > best_mf1
            print(f'    Ep {epoch+1:3d}/{RESNET_EPOCHS}  '
                  f'loss={tr_loss/len(tr_ld):.4f}  '
                  f'tr_acc={tr_correct/tr_total:.4f}  '
                  f'val_mf1={val_mf1:.4f}'
                  + ('  ← best' if is_best else ''))

        # Early stopping
        if val_mf1 > best_mf1:
            best_mf1   = val_mf1
            no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            no_improve += 1
            if no_improve >= RESNET_PATIENCE:
                if verbose:
                    print(f'    Early stop ep {epoch+1}  best_mf1={best_mf1:.4f}')
                break

    model.load_state_dict(best_state)
    if verbose:
        print(f'    ✓ ResNet best val_mf1 = {best_mf1:.4f}')
    return model


# ══════════════════════════════════════════════════════════════════
# 6. MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════
def main():
    set_global_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}\n')

    # ── Load data ────────────────────────────────────────────────────
    subject_records, _ = load_all_records(DATA_DIR)

    # ── Tạo fold groups CÙNG logic với TCN Cell 4 ─────────────
    fold_groups = build_fold_groups(subject_records, n_folds=N_FOLDS, seed=SEED)

    # ── Vòng lặp fold ────────────────────────────────────────────────
    for fold_i, test_group in enumerate(fold_groups):
        test_sids  = list(test_group)
        train_sids = [s for g in fold_groups for s in g if s not in test_sids]

        train_recs = [r for s in train_sids for r in subject_records[s]]
        test_recs  = [r for s in test_sids  for r in subject_records[s]]

        print(f'\n{"=" * 60}')
        print(f'FOLD {fold_i+1}/{N_FOLDS}')
        print(f'  Train subjects ({len(train_sids)}): {train_sids}')
        print(f'  Test  subjects ({len(test_sids)}): {test_sids}')
        print(f'  Train records: {len(train_recs)}  |  Test records: {len(test_recs)}')

        # ── Train ResNet ─────────────────────────────────────────────
        # Chú ý: train_recs KHÔNG chứa bất kỳ record nào của test_sids
        # → không có data leakage giữa ResNet và TCN
        resnet = train_resnet_for_fold(
            train_records=train_recs,
            fold_idx=fold_i,
            device=device,
        )

        # ── Lưu checkpoint — TCN Cell 12 sẽ load file này ──────
        ckpt_path = CKPT_DIR / f'resnet_fold{fold_i+1}.pth'
        torch.save(resnet.state_dict(), ckpt_path)
        print(f'  ✓ Saved → {ckpt_path}')

        # Dọn VRAM
        del resnet
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print('\n✓ Hoàn thành train ResNet cho tất cả 10 folds.')
    print(f'  Checkpoints tại: {CKPT_DIR}')


if __name__ == '__main__':
    main()


Global seed set to 123
Device: cuda

Tìm thấy 153 NPZ files
✓ 153 records  |  78 subjects

Subject-wise 10-fold CV (seed=123):
  Fold  1: [np.str_('74'), np.str_('03'), np.str_('21'), np.str_('01'), np.str_('64'), np.str_('65'), np.str_('10'), np.str_('81')]  16 rec  22,376 epochs
  Fold  2: [np.str_('77'), np.str_('54'), np.str_('15'), np.str_('71'), np.str_('40'), np.str_('49'), np.str_('31'), np.str_('38')]  16 rec  21,040 epochs
  Fold  3: [np.str_('13'), np.str_('23'), np.str_('09'), np.str_('17'), np.str_('02'), np.str_('52'), np.str_('00'), np.str_('07')]  14 rec  15,921 epochs
  Fold  4: [np.str_('34'), np.str_('53'), np.str_('59'), np.str_('32'), np.str_('48'), np.str_('20'), np.str_('66'), np.str_('55')]  16 rec  23,024 epochs
  Fold  5: [np.str_('28'), np.str_('26'), np.str_('73'), np.str_('76'), np.str_('80'), np.str_('51'), np.str_('14'), np.str_('42')]  16 rec  22,240 epochs
  Fold  6: [np.str_('04'), np.str_('30'), np.str_('56'), np.str_('19'), np.str_('18'), np.str_('67